#### RAG-Workflow Schritt für Schritt

Struktur: 

1. Websraping der Urteilstexte 
2. Abspeichern der Urteilstexte in Pandas Dataframe
3. Vektorisieren der Textinhalte durch Embedding-Model (```gemini-embedding-001```)
4. Ähnlichkeitsvergleich zwischen Frage und Urteilsabsätzen
5. Zusammenbau des Prompts
6. Antwortgenerierung durch das LLM (```gemini-3-flash-preview```)

Import der notwendigen Libraries und Konfiguration wichtiger Variablen:

In [ ]:
from __future__ import annotations

import math
#import os
from pathlib import Path
from typing import List

import pandas as pd
import requests
from bs4 import BeautifulSoup
from google import genai
from google.genai import types

EMBED_MODEL = "gemini-embedding-001"
CHAT_MODEL = "gemini-3-flash-preview"
OUTPUT_PATH = Path("bverfg_urteil_gemini.pkl")

# Der Client nutzt GEMINI_API_KEY oder GOOGLE_API_KEY aus der Umgebung.
client = genai.Client()

Funktionen zum Webscraping und Parsing des Html-Codes: 

In [ ]:
def fetch_html(url: str, timeout: int = 30) -> str:
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.text


def parse_bverfg_decision(url: str) -> pd.DataFrame:
    html = fetch_html(url)
    soup = BeautifulSoup(html, "html.parser")

    reasons_div = soup.find("div", class_="c-decision__reasons")
    if reasons_div is None:
        raise ValueError("Kein <div class='c-decision__reasons'> gefunden.")

    rows = []
    current_randnummer = None

    for tag in reasons_div.children:
        if not getattr(tag, "name", None): # Ignoriere NavigableString oder andere nicht-Tag-Elemente
            continue

        # Randnummern: <p class="is-anchor" id="8">8</p>
        if tag.name == "p" and "is-anchor" in (tag.get("class") or []): # or [] ist eine Absicherung, falls get("class") None zurückgibt
            rn_text = tag.get_text(strip=True)
            current_randnummer = int(rn_text) if rn_text.isdigit() else None # Die funktion isdigit() prüft, ob der Text eine gültige Zahl ist
            continue

        # Absatztext: <p class="justify">...</p>
        if (
            tag.name == "p"
            and "justify" in (tag.get("class") or [])
            and current_randnummer is not None
        ):
            text = tag.get_text(" ", strip=True) # Ersetzt Zeilenumbrüche durch Leerzeichen, um den Text in einer Zeile zu halten
            rows.append(
                {
                    "Text": text,
                    "Embeddings": None,
                    "Randnummer": current_randnummer,
                    "url": url,
                }
            )

    if not rows: # Wenn keine Randnummer/Text-Paare gefunden wurden, könnte das auf eine unerwartete HTML-Struktur hinweisen
        raise ValueError(
            "Keine Randnummer/Text-Paare in <div class='c-decision__reasons'> gefunden."
        )

    return pd.DataFrame(rows, columns=["Text", "Embeddings", "Randnummer", "url"])

In [ ]:
#debugging: Test der in der vorangegangenen Zelle definierten Funktion zum Scrapen und Parsen der BVerfG-Entscheidung
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
#url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2024/04/Ls20240409_2bvl000222.html?nn=68020"
#url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2024/07/fs20240730_2bvf000123.html?nn=68020"
df = parse_bverfg_decision(url)
df

Exkurs: Export als Excel-file

In [ ]:
# convert df to xlsx
output_path = Path("bverfg_urteil.xlsx")
df.to_excel(output_path, index=False)

Funktionen zur Vektorisierung der Urteils-Absätze im Dataframe

In [ ]:
def embed_texts(texts: list[str], model: str = EMBED_MODEL) -> list[list[float]]:
    """
    Erzeugt Embeddings über die Google Gen AI SDK.
    Gemini unterstützt auch mehrere Strings in einem Request.
    """
    if not texts:
        return []

    result = client.models.embed_content(
        model=model,
        contents=texts,
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY")     # hier bei gemini gibt es verschiedene Modi (SEMANTIC_SIMILARITY, RETRIEVAL_QUERY, FACT_VERIFICATION etc.)
    )

    # Laut Gemini-Dokumentation liegt das Ergebnis in result.embeddings.
    # Jedes Element enthält den Embedding-Vektor.
    embeddings = []
    for item in result.embeddings:
        if hasattr(item, "values"):
            embeddings.append(list(item.values))
        else:
            raise ValueError(f"Unerwartete Embedding-Struktur: {item!r}")

    if len(embeddings) != len(texts): # Sicherheitshalber prüfen, ob die Anzahl der Embeddings mit der Anzahl der Texte übereinstimmt.
        raise ValueError(
            f"Anzahl Embeddings ({len(embeddings)}) passt nicht zu "
            f"Anzahl Texte ({len(texts)})."
        )

    return embeddings


def add_embeddings_to_df(
    df: pd.DataFrame,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
) -> pd.DataFrame:
    """
    Fügt einem DataFrame die Embeddings hinzu.
    """
    df = df.copy() # Herstellen einer Kopie, um das Original zu sichern (nicht unbedingt notwendig)
    
    texts = df["Text"].tolist() # Alle Texte aus der Spalte "Text" werden in eine Liste gepackt, um auf sie die Funktion embed_texts() anzuwenden

    all_embeddings: list[list[float]] = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        batch_embeddings = embed_texts(batch, model=model)
        all_embeddings.extend(batch_embeddings)

    df["Embeddings"] = all_embeddings
    return df


def save_dataframe(df: pd.DataFrame, path: Path = OUTPUT_PATH) -> None:
    df.to_pickle(path)


def load_dataframe(path: Path = OUTPUT_PATH) -> pd.DataFrame:
    return pd.read_pickle(path)


def build_rag_dataframe_from_url(
    url: str,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
    path: Path = OUTPUT_PATH,
) -> pd.DataFrame:
    """
    Komplettpipeline:
    1. URL laden
    2. Absätze parsen
    3. Embeddings erzeugen
    4. Pickle speichern
    """
    df = parse_bverfg_decision(url)
    df = add_embeddings_to_df(df, model=model, batch_size=batch_size)
    save_dataframe(df, path=path)
    return df


def get_or_build_rag_dataframe( # Hauptfunktion, die prüft, ob ein Pickle mit der richtigen URL existiert, und entweder lädt oder neu erstellt
    url: str,
    model: str = EMBED_MODEL,
    batch_size: int = 32,
    path: Path = OUTPUT_PATH,
) -> pd.DataFrame:
    if path.exists():
        try:
            df = load_dataframe(path)

            if "url" in df.columns and not df.empty:
                gespeicherte_url = df["url"].iloc[0]

                if gespeicherte_url == url:
                    print("Urteil bereits vorhanden – lade DataFrame aus Pickle.")
                    return df

            print("Andere URL im Pickle gefunden – scrape und embedde neu.")
        except Exception as e:
            print(f"Pickle konnte nicht geladen werden ({e}) – scrape und embedde neu.")
    else:
        print("Noch kein Pickle vorhanden – scrape und embedde neu.")

    return build_rag_dataframe_from_url(
        url=url,
        model=model,
        batch_size=batch_size,
        path=path,
    )

In [ ]:
#debugging: Test der Komplettpipeline zum Laden, Parsen, Vektorisieruen und Speichern der BVerfG-Entscheidung
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
df = get_or_build_rag_dataframe(url, model=EMBED_MODEL, batch_size=16)
df

Funktionen zum semantischen Ähnlichkeitsvergleich: 

In [ ]:

def load_dataframe(path: Path = OUTPUT_PATH) -> pd.DataFrame:
    return pd.read_pickle(path)

def cosine_similarity(a: List[float], b: List[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot / (norm_a * norm_b)

def search_similar_passages(
    query: str,
    df: pd.DataFrame,
    model: str = EMBED_MODEL,
    top_k: int = 2,
) -> pd.DataFrame:
    if "Embeddings" not in df.columns:
        raise ValueError("Die Spalte 'Embeddings' fehlt.")

    if df["Embeddings"].isna().any():
        raise ValueError("Es fehlen Embeddings im DataFrame.")

    query_embedding = embed_texts([query], model=model)[0]

    result_df = df.copy()
    result_df["Score"] = result_df["Embeddings"].apply(
        lambda emb: cosine_similarity(query_embedding, emb)
    )

    result_df = (
        result_df.sort_values("Score", ascending=False)
        .head(top_k)
        .reset_index(drop=True)
    )

    return result_df

In [ ]:
# debugging: Test der Ähnlichkeitssuche
query = "verstößt das SolZG gegen das Grundgesetz?"
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
df = get_or_build_rag_dataframe(url, model=EMBED_MODEL, batch_size=16)

hits = search_similar_passages(query, df, model=EMBED_MODEL, top_k=10) # diese Funktion erweitert de df und gibt den erweiterten df zurück
print("\nTop-Treffer:")
hits # hits ist ein DataFrame mit den ähnlichsten Absätzen, sortiert nach Score (siehe return-Wert der Funktion search_similar_passages)

Zusammenbau des Prompt-Strings aus: 
1. Template 
2. Suchergebnissen

In [ ]:
def build_context_from_hits(hits: pd.DataFrame) -> str:
    """
    Baut aus den Top-Hits einen sauberen Kontextblock für das Sprachmodell.
    """
    context_parts = []

    for i, row in hits.iterrows():
        randnummer = row["Randnummer"]
        text = row["Text"]
        score = row.get("Score", None)

        part = (
            f"[Quelle {i + 1} | Randnummer {randnummer}"
            + (f" | Score {score:.4f}" if score is not None else "")
            + "]\n"
            f"{text}"
        )
        context_parts.append(part)

    return "\n\n".join(context_parts)


def build_rag_prompt(query: str, hits: pd.DataFrame) -> str:
    """
    Erstellt den Prompt für das Sprachmodell.
    """
    context = build_context_from_hits(hits)

    prompt = f"""
Du bist ein ein Experte für das deutsche Verfassungsrecht und spezialisiert auf die Analyse von Entscheidungen des Bundesverfassungsgerichts.

Du erhältst nachfolgend eine Frage sowie relevante Fundstellen aus einer Entscheidung des Bundesverfassungsgerichts (BVerfG), die du für die Beantwortung der Frage verwenden musst.
Beantworte die nachfolgend gestellte Frage ausschließlich auf Grundlage der unten angegebenen Fundstellen.
Wenn die Fundstellen für eine sichere Antwort nicht ausreichen, sage das ausdrücklich.
Zitiere in deiner Antwort möglichst die Randnummern in Klammern, z. B. (Rn. 12).
Beachte, dass Fundstellen in denen rechtsauffassungen im Konjunktiv dargestellt werden, nicht zwingend die Auffassung des Gerichts widerspiegeln, sondern auch Auffassung einer der Parteien sein können.

Frage:
{query}

Relevante Fundstellen:
{context}

Antworte knapp, präzise, juristisch sauber und gut strukturiert.
"""
    return prompt.strip()



In [ ]:
# debugging: Test der Komplettpipeline zum Laden, Parsen, Vektorisieren, Ähnlichketsabgleich und schließlich der Prompt-Erstellung
query = "verstößt das SolZG gegen das Grundgesetz?"
hits = search_similar_passages(query, df, model=EMBED_MODEL, top_k=10)
prompt = build_rag_prompt(query, hits)
print("\n=== PROMPT ===")
print(prompt)

Zusammenbau der Gesamt-RAG-Pipeline in einer Funktion ``` ask_rag() ```

In [ ]:
def ask_rag(
    query: str,
    url: str,
    retrieval_model: str = EMBED_MODEL,
    chat_model: str = CHAT_MODEL,
    top_k: int = 5,
    path: Path = OUTPUT_PATH,
) -> dict:
    """
    Führt Retrieval + Promptaufbau + Antwortgenerierung aus.
    Streamt die Antwort live und gibt am Ende weiterhin alles gesammelt zurück.
    """
    hits = search_similar_passages(
        query=query,
        df=get_or_build_rag_dataframe(
            url=url,
            model=retrieval_model,
            path=path,
        ),
        model=retrieval_model,
        top_k=top_k,
    )

    prompt = build_rag_prompt(query, hits)

    stream = client.models.generate_content_stream(
        model=chat_model,
        contents=prompt,
        config={
            "system_instruction": (
                "Du beantwortest Fragen zu BVerfG-Entscheidungen "
                "ausschließlich auf Basis des bereitgestellten Kontexts."
            ),
            "temperature": 0.2,
        },
    )

    answer_parts = []

    for chunk in stream:
        text = getattr(chunk, "text", None)
        if text:
            print(text, end="", flush=True)
            answer_parts.append(text)

    answer = "".join(answer_parts)

    return {
        "query": query,
        "answer": answer,
        "hits": hits,
        "prompt": prompt,
    }

In [ ]:
query = "verstößt das SolZG gegen Art. 14 GG?"
#query = "Unter welchen Voraussetzungen darf eine Ergänzungsabgabe nach Art. 106 Abs. 1 Nr. 6 GG erhoben werden?"
url = "https://www.bundesverfassungsgericht.de/SharedDocs/Entscheidungen/DE/2025/03/rs20250326_2bvr150520.html"
top_k = 15
result = ask_rag(query, url, top_k=top_k)

print("=== ANTWORT ===")
print(result["answer"])

print(f"\n=== TOP-{top_k}-HITS ===")
print(result["hits"][["Randnummer", "Score", "Text"]].to_string(index=False))